In [3]:
!pip install chromadb sentence-transformers langchain-groq
import os
import json
from typing import TypedDict, Literal, List
import chromadb
from fastapi import FastAPI
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END

# ============================================================
# CONFIGURATION
# ============================================================
BASE_DIR = os.getcwd()
DOCS_DIR = os.path.join(BASE_DIR, "docs")
CHROMA_DIR = os.path.join(BASE_DIR, "chroma_db")

# Create directories if they don't exist
os.makedirs(DOCS_DIR, exist_ok=True)
os.makedirs(CHROMA_DIR, exist_ok=True)

# Create sample policy documents
print("Creating sample policy documents...")
policy_docs = {
    "delivery_policy.txt": """Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Delivery times are typically 30-45 minutes.""",
    "return_policy.txt": """Customers can return most items within 7 days of purchase for a full refund, provided the items are unused and in their original packaging. Perishable goods are not not eligible for return.""",
    "damaged_items_policy.txt": """Any damaged items must be reported to customer support within 24 hours of delivery. Please provide photographic evidence of the damage to facilitate the refund or replacement process."""
}

for filename, content in policy_docs.items():
    with open(os.path.join(DOCS_DIR, filename), "w") as f:
        f.write(content)

print(f"Created {len(policy_docs)} sample policy documents in {DOCS_DIR}.")

COLLECTION_NAME = "zepto_policy"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# Assignment requirement:
# Unset or MOCK_LLM=1 -> deterministic offline mock mode.
# MOCK_LLM=0 -> optional real LLM mode.
MOCK_LLM = os.getenv("MOCK_LLM", "1") != "0"

# ============================================================
# INTENT KEYWORDS
# ============================================================

POLICY_KEYWORDS = [
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "cancel",
    "gift card",
    "support hours"
]

# ============================================================
# EMBEDDING MODEL
# ============================================================

print("Loading embedding model...")
embedder = SentenceTransformer(
    EMBEDDING_MODEL
)

# ============================================================
# CHROMADB
# ============================================================

print("Connecting to ChromaDB...")
# Delete the collection if it exists to ensure fresh indexing during development
try:
    chroma_client = chromadb.PersistentClient(
        path=CHROMA_DIR
    )
    chroma_client.delete_collection(name=COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")
except Exception as e:
    print(f"Collection {COLLECTION_NAME} did not exist or could not be deleted: {e}")

chroma_client = chromadb.PersistentClient(
    path=CHROMA_DIR
)
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={
        "hnsw:space": "cosine"
    }
)

# ============================================================
# LANGGRAPH STATE
# ============================================================

class GraphState(TypedDict, total=False):
    query: str
    intent: Literal[
        "policy_question",
        "general_question"
    ]
    answer: str
    sources: List[str]
    confidence: float

# ============================================================
# FASTAPI MODELS
# ============================================================

class AskRequest(BaseModel):
    query: str = Field(
        min_length=1
    )

class AskResponse(BaseModel):
    answer: str
    sources: List[str]
    confidence: float = Field(
        ge=0.0,
        le=1.0
    )

# ============================================================
# STRUCTURED PROMPT
# ============================================================

PROMPT_TEMPLATE = """
ROLE:
You are a Zepto customer-support assistant.
CONTEXT:
You are given retrieved text from Zepto's internal policy corpus.
TASK:
Answer the customer's question using only the provided retrieved context.
FORMAT:
Return JSON with exactly these fields:
{
    "answer": "string",
    "sources": ["document_id"],
    "confidence": 0.0
}

LENGTH:
Keep the answer concise, normally 1 to 3 sentences.
NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided context.
Do not invent policies, prices, timings, eligibility rules, or exceptions.

FEW-SHOT EXAMPLE:
Question:
How much is standard delivery for an order below INR 149?

Context:
Standard delivery is free on orders over INR 149; orders below this
threshold incur a flat INR 25 delivery fee.

Output:
{
    "answer": "Orders below INR 149 incur a flat INR 25 standard delivery fee.",
    "sources": ["doc_01"],
    "confidence": 1.0
}

CUSTOMER QUESTION:
{query}

RETRIEVED CONTEXT:
{context}
"""

# ============================================================
# DOCUMENT INGESTION
# ============================================================

def load_and_index_documents():
    print("Checking ChromaDB collection...")
    # Removed the 'if current_count >= 8: return' because we are ensuring
    # fresh indexing by deleting the collection before getting/creating it.
    print("Indexing policy documents...")
    for filename in sorted(
        os.listdir(DOCS_DIR)
    ):
        if not filename.endswith(".txt"):
            continue
        document_id = filename.replace(
            ".txt",
            ""
        )
        file_path = os.path.join(
            DOCS_DIR,
            filename
        )
        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as file:
            text = file.read().strip()
        # One document = one chunk.
        embedding = embedder.encode(
            text,
            normalize_embeddings=True
        ).tolist()
        collection.upsert(
            ids=[document_id],
            documents=[text],
            metadatas=[
                {
                    "document_id": document_id
                }
            ],
            embeddings=[
                embedding
            ]
        )
        print(
            f"Indexed {document_id}"
        )
    print(
        f"Total indexed documents: {collection.count()}"
    )

# ============================================================
# NODE 1: CLASSIFY INTENT
# ============================================================

def classify_intent(
    state: GraphState
) -> GraphState:
    query = state["query"]
    lowercase_query = query.lower()
    print(
        f"\nClassifying query: {query}"
    )

    # --------------------------------------------------------
    # REQUIRED MOCK MODE
    # --------------------------------------------------------

    if MOCK_LLM:
        if any(
            keyword in lowercase_query
            for keyword in POLICY_KEYWORDS
        ):
            intent = "policy_question"
        else:
            intent = "general_question"

        print(
            f"Mock classification: {intent}"
        )
        return {
            "intent": intent
        }

    # --------------------------------------------------------
    # OPTIONAL REAL LLM MODE
    # --------------------------------------------------------

    intent = real_llm_classification(
        query
    )
    return {
        "intent": intent
    }

# ============================================================
# NODE 2: RETRIEVE + ANSWER
# ============================================================

def retrieve_and_answer(
    state: GraphState
) -> GraphState:
    query = state["query"]
    print(
        "\nRetrieving documents..."
    )
    # Embed query
    query_embedding = embedder.encode(
        query,
        normalize_embeddings=True
    ).tolist()
    # Retrieve top 3
    results = collection.query(
        query_embeddings=[
            query_embedding
        ],
        n_results=3,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]
    source_ids = [
        metadata["document_id"]
        for metadata in metadatas
    ]
    print("\nRetrieved documents:")
    for document_id, distance in zip(
        source_ids,
        distances
    ):
        print(
            f"{document_id}: distance={distance:.4f}"
        )

    # --------------------------------------------------------
    # REQUIRED MOCK MODE
    # --------------------------------------------------------

    if MOCK_LLM:
        top_chunk = documents[0]
        snippet = top_chunk[:200]
        answer = (
            "Based on the retrieved context: "
            + snippet
        )
        return {
            "answer": answer,
            "sources": source_ids,
            "confidence": 1.0
        }

    # --------------------------------------------------------
    # OPTIONAL REAL LLM MODE
    # --------------------------------------------------------

    return real_llm_answer(
        query=query,
        documents=documents,
        source_ids=source_ids
    )

# ============================================================
# NODE 3: DIRECT ANSWER
# ============================================================

def direct_answer(
    state: GraphState
) -> GraphState:
    query = state["query"]
    print(
        "\nGeneral question detected."
    )

    # --------------------------------------------------------
    # REQUIRED MOCK MODE
    # --------------------------------------------------------

    if MOCK_LLM:
        return {
            "answer":
                "I can only answer questions about Zepto policies right now.",
            "sources": [],
            "confidence": 1.0
        }

    # --------------------------------------------------------
    # OPTIONAL REAL LLM MODE
    # --------------------------------------------------------

    return real_llm_direct_answer(
        query
    )

# ============================================================
# CONDITIONAL ROUTER
# ============================================================

def route_intent(
    state: GraphState
) -> str:
    return state["intent"]

# ============================================================
# BUILD LANGGRAPH
# ============================================================

def build_graph():
    graph = StateGraph(
        GraphState
    )

    # Three required nodes
    graph.add_node(
        "classify_intent",
        classify_intent
    )

    graph.add_node(
        "retrieve_and_answer",
        retrieve_and_answer
    )

    graph.add_node(
        "direct_answer",
        direct_answer
    )

    # Start
    graph.set_entry_point(
        "classify_intent"
    )

    # Conditional edge
    graph.add_conditional_edges(
        "classify_intent",
        route_intent,
        {
            "policy_question":
                "retrieve_and_answer",
            "general_question":
                "direct_answer"
        }
    )
    # End edges
    graph.add_edge(
        "retrieve_and_answer",
        END
    )

    graph.add_edge(
        "direct_answer",
        END
    )
    return graph.compile()

# ============================================================
# OPTIONAL REAL LLM HELPERS
# ============================================================

def get_groq_model():
    from langchain_groq import ChatGroq
    api_key = os.getenv(
        "GROQ_API_KEY"
    )

    if not api_key:
        raise RuntimeError(
            "GROQ_API_KEY is required when MOCK_LLM=0."
        )
    model_name = os.getenv(
        "GROQ_MODEL",
        "llama-3.1-8b-instant"
    )
    return ChatGroq(
        model=model_name,
        temperature=0,
        api_key=api_key
    )

def parse_json_response(
    text: str
):

    text = text.strip()
    if text.startswith("```"):
        text = (
            text
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )
    return json.loads(
        text
    )

def real_llm_classification(
    query: str
):

    model = get_groq_model()
    prompt = f"""
Classify the following query as exactly one of:
policy_question
general_question
Return ONLY valid JSON:
{{"intent":"policy_question"}}
Question:
{query}
"""

    for attempt in range(3):
        try:
            response = model.invoke(
                prompt
            )
            data = parse_json_response(
                response.content
            )
            intent = data["intent"]
            if intent in [
                "policy_question",
                "general_question"
            ]:
                return intent
            raise ValueError(
                "Invalid intent"
            )
        except Exception as error:
            print(
                f"LLM classification attempt "
                f"{attempt + 1} failed: {error}"
            )
            if attempt == 2:
                raise
            prompt += """
CORRECTIVE INSTRUCTION:
Return valid JSON only with exactly one
intent value: policy_question or general_question.
"""

def real_llm_answer(
    query,
    documents,
    source_ids
):
    model = get_groq_model()
    context = "\n\n".join(
        f"[{source_id}] {document}"
        for source_id, document
        in zip(
            source_ids,
            documents
        )
    )

    prompt = PROMPT_TEMPLATE.format(
        query=query,
        context=context
    )
    for attempt in range(3):
        try:
            response = model.invoke(
                prompt
            )
            data = parse_json_response(
                response.content
            )
            validated = AskResponse.model_validate(
                data
            )
            return validated.model_dump()
        except Exception as error:
            print(
                f"LLM answer attempt "
                f"{attempt + 1} failed: {error}"
            )
            if attempt == 2:
                return {
                    "answer":
                        "ERROR: real LLM output failed schema validation.",
                    "sources":
                        source_ids,
                    "confidence":
                        0.0
                }
            prompt += """
CORRECTIVE INSTRUCTION:
Return valid JSON matching exactly:
answer: string
sources: list of document IDs
confidence: float between 0 and 1
Return JSON only. Do not use markdown.
"""
def real_llm_direct_answer(
    query: str
):
    model = get_groq_model()
    prompt = f"""
Answer the following general question.
Return ONLY JSON:
{{
    "answer": "string",
    "sources": [],
    "confidence": 0.0
}}
Question:
{query}
"""
    for attempt in range(3):
        try:
            response = model.invoke(
                prompt
            )
            data = parse_json_response(
                response.content
            )
            data["sources"] = []
            validated = AskResponse.model_validate(
                data
            )
            return validated.model_dump()
        except Exception as error:
            print(
                f"LLM direct-answer attempt "
                f"{attempt + 1} failed: {error}"
            )
            if attempt == 2:
                return {
                    "answer":
                        "ERROR: real LLM output failed schema validation.",
                    "sources": [],
                    "confidence": 0.0
                }
            prompt += """
CORRECTIVE INSTRUCTION:
Return valid JSON only with answer,
sources, and confidence.
"""
# ============================================================
# INITIALIZE SYSTEM
# ============================================================
load_and_index_documents()
APP_GRAPH = build_graph()
# ============================================================
# FASTAPI APPLICATION
# ============================================================
app = FastAPI(
    title="Zepto Support Assistant",
    description="Offline RAG support assistant",
    version="1.0.0"
)
@app.get("/health")
def health():
    return {
        "status": "ok",
        "mock_llm": MOCK_LLM,
        "documents_indexed":
            collection.count()
    }
@app.post(
    "/ask",
    response_model=AskResponse
)
def ask(
    request: AskRequest
):
    result = APP_GRAPH.invoke({
        "query":
            request.query
    })
    # Final Pydantic validation
    response = AskResponse(
        answer=result["answer"],
        sources=result.get(
            "sources",
            []
        ),
        confidence=result.get(
            "confidence",
            1.0
        )
    )
    return response

Creating sample policy documents...
Created 3 sample policy documents in /content/docs.
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connecting to ChromaDB...
Deleted existing collection: zepto_policy
Checking ChromaDB collection...
Indexing policy documents...
Indexed damaged_items_policy
Indexed delivery_policy
Indexed return_policy
Total indexed documents: 3


In [6]:
import json
def run_test(query):
    result = APP_GRAPH.invoke(
        {
            "query": query
        }
    )
    print("=" * 70)
    print(
        "QUERY:"
    )
    print(
        query
    )
    print("\nJSON RESPONSE:")
    print(
        json.dumps(
            {
                "answer":
                    result["answer"],
                "sources":
                    result.get(
                        "sources",
                        []
                    ),
                "confidence":
                    result.get(
                        "confidence",
                        1.0
                    )
            },
            indent=2,
            ensure_ascii=False
        )
    )
# ============================================================
# TEST 1: POLICY QUESTION
# ============================================================
run_test(
    "What is the delivery fee for an order below INR 149?"
)
# ============================================================
# TEST 2: GENERAL QUESTION
# ============================================================
run_test(
    "What is the capital of France?"
)
# ============================================================
# TEST 3: ANOTHER POLICY QUESTION
# ============================================================
run_test(
    "How long do I have to report a damaged item?"
)


Classifying query: What is the delivery fee for an order below INR 149?
Mock classification: policy_question

Retrieving documents...

Retrieved documents:
delivery_policy: distance=0.3474
damaged_items_policy: distance=0.7877
return_policy: distance=0.8890
QUERY:
What is the delivery fee for an order below INR 149?

JSON RESPONSE:
{
  "answer": "Based on the retrieved context: Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Delivery times are typically 30-45 minutes.",
  "sources": [
    "delivery_policy",
    "damaged_items_policy",
    "return_policy"
  ],
  "confidence": 1.0
}

Classifying query: What is the capital of France?
Mock classification: general_question

General question detected.
QUERY:
What is the capital of France?

JSON RESPONSE:
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}

Classifying query: How long do I have to report a damaged i